In [ ]:
import os

from essential.gpu_utils import select_best_gpus

select_best_gpus(1)
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"


import numpy as np
import pandas as pd
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import os
import scanpy as sc
import plotnine as gg

import matplotlib.pyplot as plt
import plotly.express as px
import matplotlib.colors as mcolors
from tqdm import tqdm
import scipy.stats as st
from statsmodels.stats.multitest import multipletests

tab10_colors = plt.get_cmap("tab10").colors
tab10_hex = [mcolors.to_hex(c) for c in tab10_colors]
plt.rcParams["svg.fonttype"] = "none"

SHARED_THEME = gg.theme(
    axis_title=gg.element_text(size=7),
    axis_text=gg.element_text(size=6),
)

import sys

sys.path.append("/workspace/experiments/12312025_surrogate")

from data_resources import load_fitness_data

fitness_df = load_fitness_data()

In [ ]:
adata = sc.read_h5ad(
    "/workspace/data/251117_genomescale_CRISPRi/sample_mix_umi200_hvg500_pc25_neighbors10_mindist0.55.scvi.h5ad"
)
adata.X = adata.layers["reads"].copy()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

adata.obs["transcript_UMAP1"] = adata.obsm["X_umap"][:, 0]
adata.obs["transcript_UMAP2"] = adata.obsm["X_umap"][:, 1]

In [ ]:
adata

In [ ]:
adata.obs["spacer"].nunique()

In [ ]:
predictability_score = (
    adata.obs.groupby(["spacer"])["annotated_cluster"]
    .value_counts(normalize=True)
    .to_frame("predictability_score")
    .reset_index()
    .query("annotated_cluster == 'case-like'")
)
# predictability_score.to_csv(
#     "/workspace/data/01022026_multimodal/predictability_score.csv", index=False
# )

In [ ]:
n_capsules = adata.obs.groupby(["spacer"])["annotated_cluster"].value_counts()
n_capsules_df = (
    n_capsules.unstack(level="annotated_cluster", fill_value=0)
    .rename(
        columns={"control-like": "n_capsules_control_like", "case-like": "n_capsules_case_like"}
    )
    .reset_index()
    .assign(n_capsules=lambda x: x["n_capsules_control_like"] + x["n_capsules_case_like"])
)

spacer_trans_phenotypes = pd.merge(predictability_score, n_capsules_df, on="spacer")
spacer_trans_phenotypes["source"] = (
    "/workspace/experiments/01022026_multimodal/transcriptomic_analysis.ipynb"
)
spacer_trans_phenotypes.to_csv(
    "/workspace/data/251117_genomescale_CRISPRi/spacer_trans_phenotypes.csv", index=False
)

In [ ]:
sc.pl.umap(adata, color="annotated_cluster")

In [ ]:
adata_case = adata[adata.obs["annotated_cluster"] == "case-like"].copy()
sc.pp.neighbors(adata_case, use_rep="X_scVI")
sc.tl.umap(adata_case)
sc.tl.leiden(adata_case, key_added="leiden_case", resolution=2.0)
sc.pl.umap(adata_case, color="leiden_case")

adata_case.obs["transcript_case_UMAP1"] = adata_case.obsm["X_umap"][:, 0]
adata_case.obs["transcript_case_UMAP2"] = adata_case.obsm["X_umap"][:, 1]

In [ ]:
(
    gg.ggplot(
        adata_case.obs, gg.aes(x="transcript_case_UMAP1", y="transcript_case_UMAP2", color="T4")
    )
    + gg.geom_point(size=0.1)
    + gg.scale_color_cmap(cmap_name="bwr", limits=(-2, 2))
    + gg.theme_minimal()
    + SHARED_THEME
    + gg.theme(figure_size=(4, 3))
    + gg.labs(
        x="UMAP1",
        y="UMAP2",
        color="T4",
    )
)

In [ ]:
(
    gg.ggplot(
        adata_case.obs, gg.aes(x="transcript_case_UMAP1", y="transcript_case_UMAP2", color="T4")
    )
    + gg.geom_point(size=0.1)
    # + gg.scale_color_cmap(cmap_name="bwr", limits=(-2, 2))
    + gg.theme_minimal()
    + SHARED_THEME
    + gg.theme(figure_size=(4, 3))
    + gg.labs(
        x="UMAP1",
        y="UMAP2",
        color="T4",
    )
)

In [ ]:
fig = px.scatter(
    adata_case.obs,
    x="transcript_case_UMAP1",
    y="transcript_case_UMAP2",
    color="T4",
    color_continuous_scale="RdBu_r",  # "RdBu_r" (reversed) corresponds to "bwr" (Blue-White-Red)
    range_color=[-2, 2],
    hover_name=adata_case.obs["gene"],  # Enables hovering over the gene/obs ID
    width=400,  # Corresponds to figure_size=(4, 3) at default DPI
    height=300,
    template="plotly_white",  # Corresponds to theme_minimal
    labels={
        "transcript_case_UMAP1": "UMAP1",
        "transcript_case_UMAP2": "UMAP2",
    },
)

# Replicate the small point size and text styling
# fig.update_traces(marker=dict(size=2))
fig.update_layout(
    xaxis=dict(title_font=dict(size=7), tickfont=dict(size=6)),
    yaxis=dict(title_font=dict(size=7), tickfont=dict(size=6)),
)

fig.show()

In [ ]:
adata.obs.groupby("gene").agg({"T4": "mean"}).sort_values("T4", ascending=False)

In [ ]:
adata_case.obs.groupby("gene").agg({"T4": "mean"}).sort_values("T4", ascending=False)

In [ ]:
for leiden_cluster in range(len(adata_case.obs["leiden_case"].unique())):
    adata_case_sub = adata_case[adata_case.obs["leiden_case"] == str(leiden_cluster)].copy()
    capsule_count = adata_case_sub.obs["gene"].value_counts().loc[lambda x: x >= 2]
    # genes = capsule_count[capsule_count >= 2].index.values
    print("Cluster", leiden_cluster)
    # gene_print = ", ".join(genes)
    # print(gene_print)
    print(capsule_count)
    print()

Prompt

```
# System Role
You are an expert in *E. coli* functional genomics and metabolism. You are analyzing data from a genome-wide CRISPRi/knockdown screen with single-cell RNA-seq readout.

# Objective
Your goal is to annotate gene clusters identified in this perturbational study. For each cluster, you must identify both the **biochemical pathway** and the **physical complex** (if applicable) that best explains the set of perturbed genes.

# Guidelines
- **Granularity:** Always aim for the highest resolution possible.
    - *Good:* "Lipopolysaccharide transport", "50S Ribosomal Subunit".
    - *Bad:* "Envelope synthesis", "Translation".
- **Dual Annotation:** Provide both the biological process (pathway) and cellular component (complex) to ensure complete context.
- **Heterogeneity:** A cluster may represent a known pathway (strong signal) mixed with noise. Identify the core signal and flag outliers.

# Deliverables

## Deliverable 1: Cluster Analysis
For each cluster, provide a detailed analysis containing:

1.  **Pathway:** The specific biochemical process (e.g., "Purine biosynthesis", "Cell division").
2.  **Complex/Component:** The physical protein complex or cellular location (e.g., "DNA Polymerase III", "Inner membrane").
3.  **Consensus Label:** The single most specific name from the two above to represent this cluster (this will be used in Deliverable 2).
4.  **Gene Evidence:** A breakdown of every gene in the cluster:
    -   *Gene:* Gene symbol.
    -   *Description:* Concise description of function.
    -   *Relevance:*
        -   **Strong:** Core component of the identified pathway/complex.
        -   **Uncertain:** Functionally related but indirect (e.g., distinct operon, regulatory link).
        -   **Outlier:** No known functional connection to the cluster's main theme.

## Deliverable 2: ID Mapping
Provide a JSON object mapping the numerical Cluster ID to the **Consensus Label** identified in Deliverable 1.

{
  "0": "Consensus Label for Cluster 0",
  "1": "Consensus Label for Cluster 1"
}

# Input Data
```

In [ ]:
cluster_mapping = {
    "0": "Ribosome Biogenesis",
    "1": "Ribosome Assembly (GTPases)",
    "2": "Aerobic Respiration",
    "3": "LPS Transport System",
    "4": "Fatty Acid Biosynthesis",
    "5": "Cell Envelope Biogenesis",
    "6": "DNA Replication",
    "7": "tRNA Charging",
    "8": "Coenzyme A Biosynthesis",
    "9": "Ubiquinone Biosynthesis",
    "10": "Pyruvate Dehydrogenase Complex",
    "11": "Phospholipid Biosynthesis",
    "12": "Ribosome (Structural Components)",
    "13": "Cell Division",
    "14": "Membrane Quality Control",
    "15": "Translation Elongation",
    "16": "Aminoacyl-tRNA Biosynthesis",
    "17": "DNA Replication Initiation",
    "18": "SRP Pathway",
    "19": "Transcription Termination",
    "20": "RNA Processing",
    "21": "PTS System",
    "22": "Pyridoxal Phosphate Biosynthesis",
    "23": "Phosphate Transport",
    "24": "Methionine Biosynthesis",
    "25": "RNA Degradation",
    "26": "Carbon Storage Regulation",
}

cluster_mapping_coarse = {
    "0": "Ribosome Biogenesis",
    "1": "Ribosome Biogenesis",
    "2": "Metabolism & Energetics",
    "3": "Cell Envelope & Membrane Biology",
    "4": "Cell Envelope & Membrane Biology",
    "5": "Cell Envelope & Membrane Biology",
    "6": "DNA Replication",
    "7": "Ribosome Translation",
    "8": "Metabolism & Energetics",
    "9": "Metabolism & Energetics",
    "10": "Metabolism & Energetics",
    "11": "Cell Envelope & Membrane Biology",
    "12": "Ribosome Biogenesis",
    "13": "Cell Division",
    "14": "Cell Envelope & Membrane Biology",
    "15": "Ribosome Translation",
    "16": "Ribosome Translation",
    "17": "DNA Replication",
    "18": "Cell Envelope & Membrane Biology",
    "19": "Transcription & RNA Processing",
    "20": "Ribosome Biogenesis",
    "21": "Metabolism & Energetics",
    "22": "Metabolism & Energetics",
    "23": "Metabolism & Energetics",
    "24": "Metabolism & Energetics",
    "25": "Transcription & RNA Processing",
    "26": "Metabolism & Energetics",
}


**Cluster 0**
1.  **Pathway:** Ribosome Biogenesis (rRNA processing and subunit assembly).
2.  **Complex/Component:** Ribosome Assembly Factors / 50S Subunit.
3.  **Consensus Label:** **Ribosome Biogenesis**.
4.  **Gene Evidence:**
    *   **nusB** (9), **suhB** (5): **Strong**. Transcription antitermination factors involved in rRNA synthesis.
    *   **rluD** (7), **rnc** (7), **rbfA** (4), **rimM** (2): **Strong**. rRNA processing and modification enzymes (Pseudouridine synthase, RNase III, Binding Factor A).
    *   **ribE** (12): **Outlier/Uncertain**. Riboflavin synthase. Essential metabolic gene, possibly clustering due to high fitness cost or operon structure, though functionally distinct.
    *   **iscU, iscX, iscS, hscA, hscB, fdx** (~2-4): **Outlier**. Iron-Sulfur Cluster (ISC) assembly. Essential for some ribosomal modifications (e.g., *rimO*), but represents a distinct functional module mixed in this cluster.

**Cluster 1**
1.  **Pathway:** Ribosome Assembly (GTPase-mediated).
2.  **Complex/Component:** Ribosomal Subunits (30S/50S) & Assembly GTPases.
3.  **Consensus Label:** **Ribosome Assembly (GTPases)**.
4.  **Gene Evidence:**
    *   **der** (10), **era** (8), **obgE** (8): **Strong**. Essential GTPases required for 50S and 30S ribosomal subunit maturation.
    *   **rplA, rplC, rpsB, infB**: **Strong**. Primary ribosomal proteins and Initiation Factor 2, central to translation initiation and assembly.
    *   **hemA** (8): **Uncertain**. Glutamyl-tRNA reductase (Heme biosynthesis). Uses tRNA-Glu, linking it to translation capacity.

**Cluster 2**
1.  **Pathway:** Aerobic Respiration / Electron Transport Chain.
2.  **Complex/Component:** Cytochrome *bo3* Ubiquinol Oxidase.
3.  **Consensus Label:** **Aerobic Respiration**.
4.  **Gene Evidence:**
    *   **cyoA, cyoB, cyoC, cyoD, cyoE**: **Strong**. Subunits of the Cytochrome *bo3* complex.
    *   **hemE, hemB**: **Strong**. Heme biosynthesis (cofactor for cytochromes).
    *   **ubiG**: **Strong**. Ubiquinone biosynthesis (electron carrier).

**Cluster 3**
1.  **Pathway:** Lipopolysaccharide (LPS) Transport.
2.  **Complex/Component:** Lpt Complex / Lol Complex.
3.  **Consensus Label:** **LPS Transport System**.
4.  **Gene Evidence:**
    *   **lolE, lolD, lolA, lolB, lolC**: **Strong**. Lipoprotein localization machinery.
    *   **lptF, lptD, lptG, lptB**: **Strong**. LPS transport machinery (trans-envelope bridge).
    *   **kdsC, lpxA**: **Uncertain**. LPS synthesis (upstream of transport), likely co-clustered due to envelope stress.

**Cluster 4**
1.  **Pathway:** Fatty Acid and LPS Core Biosynthesis.
2.  **Complex/Component:** Fatty Acid Synthase (FAS II).
3.  **Consensus Label:** **Fatty Acid Biosynthesis**.
4.  **Gene Evidence:**
    *   **fabH, fabI, fabD, fabG, acpP**: **Strong**. Core components of type II fatty acid synthesis.
    *   **waaA, kdsB, kdsA, lpxK**: **Strong**. Biosynthesis of the KDO-Lipid A and LPS core oligosaccharide (requires fatty acids).
    *   **accD, accC**: **Strong**. Acetyl-CoA carboxylase subunits (lipid precursor synthesis).

**Cluster 5**
1.  **Pathway:** Envelope Stress Response / Cell Wall Biogenesis.
2.  **Complex/Component:** Rcs Phosphorelay / Outer Membrane.
3.  **Consensus Label:** **Cell Envelope Biogenesis**.
4.  **Gene Evidence:**
    *   **yrfF (igaA)** (4): **Strong**. Negative regulator of the Rcs phosphorelay (essential for preventing constitutive stress response).
    *   **bamA** (2): **Strong**. Beta-barrel assembly machine (OMP insertion).
    *   **murC**: **Uncertain**. Peptidoglycan synthesis.

**Cluster 6**
1.  **Pathway:** DNA Replication and dNTP Synthesis.
2.  **Complex/Component:** DNA Polymerase III Holoenzyme.
3.  **Consensus Label:** **DNA Replication**.
4.  **Gene Evidence:**
    *   **holA, holB, dnaX, dnaN**: **Strong**. Components of the DNA Polymerase III clamp loader and sliding clamp.
    *   **nrdA, nrdB**: **Strong**. Ribonucleotide reductase (dNTP biosynthesis).
    *   **folA, folC, thyA**: **Strong**. Folate metabolism required for dTTP synthesis.

**Cluster 7**
1.  **Pathway:** Amino Acid Transport and Charging.
2.  **Complex/Component:** Aminoacyl-tRNA Synthetases (Class II).
3.  **Consensus Label:** **tRNA Charging**.
4.  **Gene Evidence:**
    *   **pheS, pheT, asnS, lysS, cysS, thrS**: **Strong**. Aminoacyl-tRNA synthetases.
    *   **gcvR**: **Uncertain**. Glycine cleavage system regulator (amino acid metabolism).

**Cluster 8**
1.  **Pathway:** Coenzyme A Biosynthesis / Glycolysis.
2.  **Complex/Component:** CoA Biosynthetic Complex.
3.  **Consensus Label:** **Coenzyme A Biosynthesis**.
4.  **Gene Evidence:**
    *   **dfp (coaBC), coaD**: **Strong**. Essential enzymes in CoA biosynthesis.
    *   **eno, gapA**: **Uncertain**. Glycolytic enzymes. Often essential/high abundance.

**Cluster 9**
1.  **Pathway:** Ubiquinone (Coenzyme Q8) Biosynthesis.
2.  **Complex/Component:** Cytosol / Inner Membrane.
3.  **Consensus Label:** **Ubiquinone Biosynthesis**.
4.  **Gene Evidence:**
    *   **ubiB, ubiJ, ubiE, ubiG**: **Strong**. Dedicated ubiquinone biosynthesis enzymes.
    *   **ispA**: **Strong**. Farnesyl diphosphate synthase (isoprenoid precursor for quinones).
    *   **hemC, hemE**: **Uncertain**. Heme synthesis (overlap with quinone pathway regulation).

**Cluster 10**
1.  **Pathway:** Pyruvate Oxidation / Acetyl-CoA Synthesis.
2.  **Complex/Component:** Pyruvate Dehydrogenase Complex (PDH).
3.  **Consensus Label:** **Pyruvate Dehydrogenase Complex**.
4.  **Gene Evidence:**
    *   **aceE** (7): **Strong**. Pyruvate dehydrogenase E1 subunit.
    *   **lipA** (7): **Strong**. Lipoic acid synthase (essential cofactor for AceE).
    *   **ackA, pta**: **Uncertain**. Acetate metabolism (downstream of Acetyl-CoA).

**Cluster 11**
1.  **Pathway:** Phospholipid Biosynthesis.
2.  **Complex/Component:** Inner Membrane.
3.  **Consensus Label:** **Phospholipid Biosynthesis**.
4.  **Gene Evidence:**
    *   **psd** (5), **pssA**: **Strong**. Phosphatidylserine decarboxylase/synthase (PE synthesis).
    *   **lptD, skp, mnmA**: **Uncertain**. Envelope/LPS related.

**Cluster 12**
1.  **Pathway:** Translation.
2.  **Complex/Component:** Ribosome (Structural Subunits).
3.  **Consensus Label:** **Ribosome (Structural Components)**.
4.  **Gene Evidence:**
    *   **rpsE, rplC, rplF, rpmD, rplX**: **Strong**. Structural proteins of the 30S and 50S subunits.
    *   **recA**: **Outlier**. DNA repair.

**Cluster 13**
1.  **Pathway:** Cell Division (Septation).
2.  **Complex/Component:** Divisome / Z-ring.
3.  **Consensus Label:** **Cell Division**.
4.  **Gene Evidence:**
    *   **ftsZ, ftsQ, ftsA, zipA**: **Strong**. Core components of the cell division machinery.
    *   **ddlB, mur** genes: **Strong**. Peptidoglycan synthesis at the septum.

**Cluster 14**
1.  **Pathway:** Membrane Protein Quality Control / Lipid Biosynthesis.
2.  **Complex/Component:** FtsH Protease Complex.
3.  **Consensus Label:** **Membrane Quality Control**.
4.  **Gene Evidence:**
    *   **ftsH** (8): **Strong**. Essential membrane protease, regulates LpxC (LPS synthesis).
    *   **lpxC, lpxK, fabB**: **Strong**. Lipid/LPS synthesis targets regulated by FtsH.
    *   **secB**: **Strong**. Chaperone for protein export.

**Cluster 15**
1.  **Pathway:** Translation Elongation.
2.  **Complex/Component:** Elongation Factor Complex.
3.  **Consensus Label:** **Translation Elongation**.
4.  **Gene Evidence:**
    *   **tufA, tufB** (EF-Tu), **tsf** (EF-Ts): **Strong**. The core elongation factors.
    *   **gly, hisS, glnS**: **Uncertain**. tRNAs and synthetases.

**Cluster 16**
1.  **Pathway:** Amino Acid Biosynthesis / Stringent Response.
2.  **Complex/Component:** Cytosol.
3.  **Consensus Label:** **Aminoacyl-tRNA Biosynthesis**.
4.  **Gene Evidence:**
    *   **serS, lysS, proS, trpS**: **Strong**. Aminoacyl-tRNA synthetases.
    *   **poxB**: **Outlier**. Pyruvate oxidase (stationary phase survival).
    *   **nadA**: **Outlier**. NAD biosynthesis.

**Cluster 17**
1.  **Pathway:** DNA Replication Initiation / Elongation.
2.  **Complex/Component:** Primosome / Replisome.
3.  **Consensus Label:** **DNA Replication Initiation**.
4.  **Gene Evidence:**
    *   **dnaE, dnaC, dnaB**: **Strong**. DNA polymerase III alpha, helicase loader, and helicase.
    *   **racR**: **Outlier**. Prophage repressor (likely strain-specific essentiality).

**Cluster 18**
1.  **Pathway:** Protein Export.
2.  **Complex/Component:** Signal Recognition Particle (SRP).
3.  **Consensus Label:** **SRP Pathway**.
4.  **Gene Evidence:**
    *   **ftsY, ffh**: **Strong**. SRP receptor and protein.
    *   **secY**: **Strong**. Translocon channel.

**Cluster 19**
1.  **Pathway:** Transcription Termination.
2.  **Complex/Component:** Transcription Elongation Complex.
3.  **Consensus Label:** **Transcription Termination**.
4.  **Gene Evidence:**
    *   **rho**: **Strong**. Transcription termination factor.
    *   **nusG**: **Strong**. Transcription elongation/termination factor.

**Cluster 20**
1.  **Pathway:** RNA Processing / Antitermination.
2.  **Complex/Component:** RNase P / NusA Complex.
3.  **Consensus Label:** **RNA Processing**.
4.  **Gene Evidence:**
    *   **rnpA, rnpB**: **Strong**. Protein and RNA components of RNase P.
    *   **nusA**: **Strong**. Transcription pausing/antitermination.

**Cluster 21**
1.  **Pathway:** Carbohydrate Transport.
2.  **Complex/Component:** Phosphotransferase System (PTS).
3.  **Consensus Label:** **PTS System**.
4.  **Gene Evidence:**
    *   **ptsH, ptsI**: **Strong**. General PTS components (HPr, Enzyme I).
    *   **pyrH**: **Outlier**. UMP kinase.

**Cluster 22**
1.  **Pathway:** Vitamin B6 Biosynthesis.
2.  **Complex/Component:** Cytosol.
3.  **Consensus Label:** **Pyridoxal Phosphate Biosynthesis**.
4.  **Gene Evidence:**
    *   **pdxA, pdxJ**: **Strong**. Pyridoxal phosphate biosynthetic enzymes.

**Cluster 23**
1.  **Pathway:** Phosphate Transport.
2.  **Complex/Component:** PstSCAB Complex.
3.  **Consensus Label:** **Phosphate Transport**.
4.  **Gene Evidence:**
    *   **pstA, pstB, pstC**: **Strong**. Phosphate ABC transporter subunits.
    *   **phoU**: **Strong**. Phosphate transport regulator.

**Cluster 24**
1.  **Pathway:** Methionine Metabolism.
2.  **Complex/Component:** Cytosol.
3.  **Consensus Label:** **Methionine Biosynthesis**.
4.  **Gene Evidence:**
    *   **metJ, metK**: **Strong**. Methionine repressor and SAM synthase.

**Cluster 25**
1.  **Pathway:** RNA Degradation.
2.  **Complex/Component:** RNA Degradosome.
3.  **Consensus Label:** **RNA Degradation**.
4.  **Gene Evidence:**
    *   **rne**: **Strong**. RNase E (essential endonuclease).
    *   **rnhA**: **Strong**. RNase HI.

**Cluster 26**
1.  **Pathway:** Carbon Storage Regulation.
2.  **Complex/Component:** Csr System.
3.  **Consensus Label:** **Carbon Storage Regulation**.
4.  **Gene Evidence:**
    *   **csrA**: **Strong**. RNA-binding protein, global regulator of carbon metabolism.



In [ ]:
adata_case.obs["annotated_leiden_case"] = adata_case.obs["leiden_case"].map(cluster_mapping)
adata_case.obs["annotated_leiden_case_coarse"] = adata_case.obs["leiden_case"].map(
    cluster_mapping_coarse
)
adata_case.obs["source"] = (
    "/workspace/experiments/01022026_multimodal/transcriptomic_analysis.ipynb"
)
adata_case.write_h5ad("/workspace/data/251117_genomescale_CRISPRi/adata_case.annotated.h5ad")

In [ ]:
fig = px.scatter(
    adata_case.obs,
    x="transcript_case_UMAP1",
    y="transcript_case_UMAP2",
    color="annotated_leiden_case",
    hover_data=["target"],
)
fig.update_traces(marker=dict(size=3))
fig.update_layout(template="plotly_white", width=1000, height=700)

fig.show()

In [ ]:
sc.pl.umap(adata_case, color="annotated_leiden_case_coarse")

In [ ]:
plot_df = adata_case.obs.copy()
plot_df_avg = (
    plot_df.groupby("annotated_leiden_case")[["transcript_case_UMAP1", "transcript_case_UMAP2"]]
    .mean()
    .reset_index()
)
fig = (
    gg.ggplot(
        adata_case.obs,
    )
    + gg.geom_point(
        gg.aes(x="transcript_case_UMAP1", y="transcript_case_UMAP2", color="annotated_leiden_case"),
        size=1.0,
    )
    + gg.geom_text(
        gg.aes(x="transcript_case_UMAP1", y="transcript_case_UMAP2", label="annotated_leiden_case"),
        data=plot_df_avg,
        size=10,
    )
    + gg.theme_minimal()
    + gg.labs(
        x="UMAP1",
        y="UMAP2",
    )
    + gg.theme(figure_size=(10, 10), legend_position="none")
)
fig.save("transcriptomic_umap_case_leiden.svg")
fig

Here is a comprehensive annotation of the identified *E. coli* functional modules, organized by cellular process.

***

Functional Annotation of *E. coli* Essential Processes

This document details the biological roles of gene clusters identified in the CRISPRi perturbation screen. The clusters are grouped by their physiological context, distinguishing between information processing (Central Dogma), cell envelope biogenesis, and central metabolism.

I. Information Processing: The Central Dogma

This super-group contains the machinery responsible for maintaining the genome and converting genetic information into functional proteins. In fast-growing *E. coli*, these processes account for the majority of cellular energy expenditure.

A. DNA Replication
DNA replication in *E. coli* is bidirectional, initiating at *oriC*. It requires the coordination of unwinding DNA (helicase), synthesizing primers (primase), and rapid polymerization.

*   **Cluster 17: DNA Replication Initiation (Primosome)**
    *   **Context:** Before DNA synthesis can begin, the double helix must be opened and loaded with the replicative machinery.
    *   **Mechanism:** **DnaA** (not in cluster, initiator) recruits **DnaC** (loader), which loads the **DnaB** helicase onto the DNA. The helicase unwinds DNA, allowing **DnaG** (primase, seen in Cluster 6) to synthesize RNA primers. **DnaE** is the alpha subunit of Polymerase III, the main replicative enzyme.
*   **Cluster 6: DNA Replication (Pol III Holoenzyme)**
    *   **Context:** High-speed DNA elongation requires the Polymerase III Holoenzyme.
    *   **Mechanism:** The "clamp loader" complex (**HolA, HolB, DnaX**) loads the beta-sliding clamp (**DnaN**) onto DNA, tethering the polymerase for high processivity.
    *   **Precursors:** This cluster also includes **NrdA/NrdB** (Ribonucleotide Reductase), which converts ribonucleotides (NTPs) to deoxyribonucleotides (dNTPs), the fundamental building blocks of DNA. **ThyA** and **FolA** are specific for dTTP synthesis (thymine manufacturing).

B. Transcription & RNA Processing
Transcription involves synthesizing RNA from a DNA template. In bacteria, this is tightly coupled with translation, but specific machinery handles termination and processing.

*   **Cluster 19: Transcription Termination**
    *   **Context:** Transcription must stop at precise locations to prevent interference with downstream genes.
    *   **Mechanism:** **Rho** is an ATP-dependent helicase that chases the RNA polymerase and dislodges it at specific "rut" sites. **NusG** links transcription and translation, preventing Rho from acting prematurely on translated mRNAs.
*   **Cluster 20: RNA Processing**
    *   **Context:** Nascent RNAs often require processing to become functional (e.g., removing leader sequences).
    *   **Mechanism:** **RnpA/RnpB** form RNase P, a ribozyme responsible for cleaving the 5' leader sequence of tRNA precursors. **NusA** modulates elongation rate and pausing, critical for proper RNA folding and Rho-dependent termination.
*   **Cluster 25: RNA Degradation**
    *   **Context:** mRNA turnover allows rapid adaptation to new environments.
    *   **Mechanism:** **Rne** (RNase E) is the primary endonuclease that initiates mRNA decay and processes rRNA/tRNA precursors. It serves as the scaffold for the "RNA degradosome."

C. Translation: Ribosome Biogenesis
The ribosome is a massive ribonucleoprotein complex (70S) composed of a small (30S) and large (50S) subunit. Its assembly is complex and consumes significant cellular resources.

*   **Cluster 0: Ribosome Biogenesis**
    *   **Context:** Maturation of the ribosomal RNA (rRNA) scaffold.
    *   **Mechanism:** **NusB** ensures the complete transcription of the long *rrn* operons (antitermination). **Rnc** (RNase III) cleaves the initial rRNA transcript into precursors. **RluD** modifies specific uridine bases to pseudouridine in the 23S rRNA, stabilizing the ribosome core.
*   **Cluster 1: Ribosome Assembly (GTPases)**
    *   **Context:** Assembly checkpoints.
    *   **Mechanism:** A dedicated set of essential GTPases (**Der, Era, ObgE**) acts as quality control checkpoints. They bind to immature ribosomal subunits in a GTP-dependent manner, preventing them from entering the translation pool until they are properly folded.
*   **Cluster 12: Ribosome (Structural Components)**
    *   **Context:** The physical bricks of the ribosome.
    *   **Mechanism:** This cluster contains structural proteins (e.g., **RplC, RpsE**) that bind directly to rRNA to form the architecture of the 30S and 50S subunits.

D. Translation: Active Protein Synthesis
Once assembled, ribosomes require factors to initiate synthesis, elongate the peptide chain, and terminate.

*   **Cluster 7 & 16: tRNA Charging (Aminoacyl-tRNA Synthetases)**
    *   **Context:** The "genetic code" is physically instantiated by these enzymes, which attach specific amino acids to their corresponding tRNAs.
    *   **Mechanism:** Cluster 7 contains mostly Class II synthetases (**PheS/T, AsnS, LysS**) while Cluster 16 contains others (**SerS, ProS**). Without these, translation stalls due to a lack of charged tRNAs.
*   **Cluster 15: Translation Elongation**
    *   **Context:** The rapid addition of amino acids to the growing chain.
    *   **Mechanism:** **TufA/B** (EF-Tu) delivers charged tRNAs to the ribosome. **Tsf** (EF-Ts) recycles EF-Tu by swapping its GDP for GTP. This cycle is the most frequent enzymatic reaction in the cell.

II. The Cell Envelope: Defense and Division

*E. coli* is a Gram-negative bacterium, meaning it has two membranes (inner and outer) separated by a periplasmic space containing the peptidoglycan cell wall.

A. Lipid and Membrane Synthesis
*   **Cluster 4: Fatty Acid Biosynthesis**
    *   **Context:** The hydrophobic tails of membrane lipids.
    *   **Mechanism:** The Type II Fatty Acid Synthase (FAS II) system. **FabH** initiates the chain, and the cycle of **FabG, FabZ, FabI** elongates it. **AccD** provides the malonyl-CoA precursors.
*   **Cluster 11: Phospholipid Biosynthesis**
    *   **Context:** Assembling the lipid bilayer.
    *   **Mechanism:** **Psd** and **PssA** synthesize phosphatidylethanolamine (PE), the major phospholipid in *E. coli* membranes, from serine and CDP-diacylglycerol.
*   **Cluster 3: LPS Transport System**
    *   **Context:** Lipopolysaccharide (LPS) is the unique, protective outer layer of the outer membrane. It is synthesized in the inner membrane and must be transported across the periplasm.
    *   **Mechanism:** The **Lpt** machinery (LptBFG at inner membrane, LptD at outer membrane) forms a physical bridge to push hydrophobic LPS molecules across the aqueous periplasm to the cell surface.

B. Protein Targeting and Quality Control
*   **Cluster 18: SRP Pathway**
    *   **Context:** Targeting membrane proteins to the inner membrane.
    *   **Mechanism:** The Signal Recognition Particle (**Ffh** + 4.5S RNA) recognizes hydrophobic signal sequences as they emerge from the ribosome and docks them to the **FtsY** receptor at the membrane, allowing co-translational insertion via **SecY**.
*   **Cluster 14: Membrane Quality Control**
    *   **Context:** Removing misfolded membrane proteins and regulating lipid balance.
    *   **Mechanism:** **FtsH** is an ATP-dependent membrane protease. Crucially, it degrades **LpxC** (the committing step of LPS synthesis) to balance phospholipid and LPS production.

C. Cell Division
*   **Cluster 13: Cell Division**
    *   **Context:** Cytokinesis—splitting one cell into two.
    *   **Mechanism:** **FtsZ** forms a contractile ring (Z-ring) at mid-cell. **FtsA** and **ZipA** anchor this ring to the membrane. **FtsQ** is part of the divisome complex that coordinates septal peptidoglycan synthesis (**DdlB**) to build the new cell poles.

III. Metabolism and Energetics

A. Energy Generation
*   **Cluster 2: Aerobic Respiration**
    *   **Context:** Generating the Proton Motive Force (PMF) using oxygen as the terminal electron acceptor.
    *   **Mechanism:** The Cytochrome *bo3* ubiquinol oxidase (**CyoA-E**) pumps protons out of the cell while reducing oxygen to water. It requires Heme (**HemE**) and Ubiquinone (**UbiG**) cofactors.
*   **Cluster 10: Pyruvate Dehydrogenase Complex (PDH)**
    *   **Context:** The link between Glycolysis and the TCA cycle (respiration).
    *   **Mechanism:** Converts Pyruvate to Acetyl-CoA + CO2. **AceE** is the E1 decarboxylase component; **LipA** synthesizes the essential lipoate cofactor required for this reaction.
*   **Cluster 9: Ubiquinone Biosynthesis**
    *   **Context:** The electron carrier shuttling electrons between dehydrogenases and oxidases in the membrane.
    *   **Mechanism:** **Ubi** genes perform a series of modifications (methylation, hydroxylation) on the aromatic ring of the quinone precursor.

B. Transport and Regulation
*   **Cluster 21: PTS System**
    *   **Context:** Importing sugars (usually glucose) and phosphorylating them simultaneously.
    *   **Mechanism:** **PtsI** (Enzyme I) and **PtsH** (HPr) are the general components that transfer phosphate from PEP to sugar-specific permeases. This system also regulates global carbon catabolite repression.
*   **Cluster 23: Phosphate Transport**
    *   **Context:** Acquiring inorganic phosphate (Pi) for ATP, DNA, and RNA.
    *   **Mechanism:** The **PstSCAB** is a high-affinity ABC transporter. **PhoU** regulates the uptake to prevent toxicity.
*   **Cluster 26: Carbon Storage Regulation**
    *   **Context:** Switching between glycolysis and gluconeogenesis/storage.
    *   **Mechanism:** **CsrA** is an RNA-binding protein that inhibits the translation of glycogen synthesis genes and activates glycolysis (e.g., via *pgm*), acting as a global switch for carbon flux.

C. Cofactor Biosynthesis
*   **Cluster 8: Coenzyme A Biosynthesis**
    *   **Context:** CoA is the universal acyl carrier (e.g., Acetyl-CoA, Succinyl-CoA).
    *   **Mechanism:** **Dfp** (CoaBC) catalyzes the decarboxylation of pantothenate (Vitamin B5) intermediates to form CoA.
*   **Cluster 22: Pyridoxal Phosphate Biosynthesis**
    *   **Context:** PLP (Vitamin B6) is the essential cofactor for amino acid transaminases.
    *   **Mechanism:** **PdxA** and **PdxJ** synthesize the PLP ring structure.

***

Gap Analysis: What is Missing?

While the clusters cover a vast array of essential biology, several key processes typically essential or highly active in *E. coli* are notably absent or underrepresented in distinct clusters:

1.  **Peptidoglycan (Cell Wall) Synthesis:** While we see *murC* (Cluster 5) and *ddlB* (Cluster 13), the core "Mur" pathway (MurA-F) and the shape-determining elongation complex (MreB, PBP2) do not form a distinct, cohesive cluster. They may be dispersed or their signal was too weak/heterogeneous.
2.  **Transcription Initiation (Sigma Factors):** We see the core RNA polymerase termination factors, but the primary sigma factor **RpoD** (Sigma 70), which directs polymerase to promoters, is not the center of a distinct cluster.
3.  **Core Glycolysis and TCA Cycle:** While we see PDH (Cluster 10) and Respiration (Cluster 2), the enzymes for Glycolysis (GapA, Eno are present but scattered) and the full TCA cycle (Citrate synthase, etc.) are not grouped as a cohesive "Central Carbon Metabolism" unit. This is common in rich media where auxotrophy is rescued, but glycolysis usually remains essential.
4.  **Amino Acid Biosynthesis:** We see tRNA charging (Synthetases), but not the actual synthesis pathways for amino acids (e.g., the *trp*, *his*, or *ilv* operons). This strongly suggests the experiment was performed in **Rich Media (e.g., LB)**, where amino acids are imported rather than synthesized, rendering biosynthetic genes non-essential.
5.  **SecYEG Translocon:** While we see *secY* in Cluster 18 (SRP), the Sec translocon often forms its own strong cluster involving *secA*, *secE*, *secG*. *secE* is in Cluster 19, suggesting some splitting of the translocation machinery.

# Gene visualizations 

## Mur 

In [ ]:
adata_sub = adata_case.obs.loc[lambda x: x["target"].astype(str).str.startswith("mur")].copy()
adata_sub["target"] = adata_sub["target"].astype(str)

(
    gg.ggplot(
        adata_case.obs,
    )
    + gg.geom_point(
        gg.aes(x="transcript_case_UMAP1", y="transcript_case_UMAP2"),
        size=0.5,
    )
    + gg.geom_point(
        adata_sub,
        gg.aes(x="transcript_case_UMAP1", y="transcript_case_UMAP2", color="target"),
        size=3,
    )
    + gg.theme_minimal()
    + gg.theme(figure_size=(12, 10))
)

## Rpo

In [ ]:
adata_sub = adata_case.obs.loc[lambda x: x["target"].astype(str).str.startswith("rpo")].copy()
adata_sub["target"] = adata_sub["target"].astype(str)

(
    gg.ggplot(
        adata_case.obs,
    )
    + gg.geom_point(
        gg.aes(x="transcript_case_UMAP1", y="transcript_case_UMAP2"),
        size=0.5,
    )
    + gg.geom_point(
        adata_sub,
        gg.aes(x="transcript_case_UMAP1", y="transcript_case_UMAP2", color="target"),
        size=3,
    )
    + gg.theme_minimal()
    + gg.theme(figure_size=(12, 10))
)

In [ ]:
(
    gg.ggplot(adata.obs, gg.aes(x="transcript_UMAP1", y="transcript_UMAP2"))
    + gg.geom_point(size=0.5)
    + gg.geom_point(
        adata.obs.loc[lambda x: x["target"].astype(str).str.startswith("rpo")].assign(
            target=lambda x: x["target"].astype(str)
        ),
        gg.aes(color="target"),
        size=3,
    )
    + gg.theme_minimal()
    + gg.theme(figure_size=(12, 10))
)

In [ ]:
(
    gg.ggplot(adata.obs.loc[lambda x: x["target"].astype(str).str.startswith("rpo")])
    + gg.geom_boxplot(gg.aes(x="annotated_cluster", y="n_counts"))
    + gg.geom_jitter(gg.aes(x="annotated_cluster", y="n_counts"), size=0.5)
    + gg.labs(x="Cluster", y="Number of counts")
    + gg.theme_minimal()
)

## lpx pathway

In [ ]:
adata_sub = adata_case.obs.loc[lambda x: x["target"].isin(["lpxA", "lpxB", "lpxD", "lpxK"])].copy()
adata_sub["target"] = pd.Categorical(
    adata_sub["target"], categories=["lpxA", "lpxB", "lpxD", "lpxK"]
)

(
    gg.ggplot(
        adata_case.obs,
    )
    + gg.geom_point(
        gg.aes(x="transcript_case_UMAP1", y="transcript_case_UMAP2"),
        size=0.5,
    )
    + gg.geom_point(
        adata_sub,
        gg.aes(x="transcript_case_UMAP1", y="transcript_case_UMAP2", color="target"),
        size=3,
    )
    + gg.theme_minimal()
    + gg.theme(figure_size=(12, 10))
)

In [ ]:
fitness_df_ = fitness_df.loc[lambda x: x["gene"].isin(["lpxA", "lpxB", "lpxD", "lpxK"])]
fitness_df_["gene"] = pd.Categorical(
    fitness_df_["gene"], categories=["lpxA", "lpxB", "lpxD", "lpxK"]
)
(
    gg.ggplot(fitness_df_, gg.aes(x="gene", y="T4", fill="gene"))
    + gg.geom_boxplot()
    + gg.geom_jitter()
)

# Detectable vs non-detectable genes

#### Poor efficiency

In [ ]:
adata_phenotype = adata[adata.obs["annotated_cluster"] == "case-like"].copy()
adata_nophenotype = adata[adata.obs["annotated_cluster"] != "case-like"].copy()

plot_df = []
for gene in tqdm(adata_phenotype.obs["gene"].unique()):
    try:
        x_phenotype = (
            adata_phenotype[adata_phenotype.obs["gene"] == gene, gene]
            .layers["reads"]
            .toarray()
            .flatten()
        ).mean()
        x_cp10k_phenotype = (
            adata_phenotype[adata_phenotype.obs["gene"] == gene, gene]
            .layers["cp10k"]
            .toarray()
            .flatten()
        ).mean()

        x_nophenotype = (
            adata_nophenotype[adata_nophenotype.obs["gene"] == gene, gene]
            .layers["reads"]
            .toarray()
            .flatten()
        ).mean()
        x_cp10k_nophenotype = (
            adata_nophenotype[adata_nophenotype.obs["gene"] == gene, gene]
            .layers["cp10k"]
            .toarray()
            .flatten()
        ).mean()

        fc_ = (x_phenotype + 1e-6) / (x_nophenotype + 1e-6)
        log2fc_ = np.log2(fc_)

        plot_df.append(
            {
                "gene": gene,
                "log2fc": log2fc_,
                "fc": fc_,
                "x_phenotype": x_phenotype,
                "x_nophenotype": x_nophenotype,
                "x_cp10k_phenotype": x_cp10k_phenotype,
                "x_cp10k_nophenotype": x_cp10k_nophenotype,
            }
        )
    except Exception as e:
        print(e)
        continue

plot_df = pd.DataFrame(plot_df)

In [ ]:
plot_df_ = plot_df[["x_phenotype", "x_nophenotype"]].dropna(axis=0)
from scipy.stats import wilcoxon

print(wilcoxon(plot_df_["x_phenotype"], plot_df_["x_nophenotype"]))

(
    gg.ggplot(plot_df, gg.aes(x="x_phenotype", y="x_nophenotype"))
    + gg.geom_point()
    + gg.theme_minimal()
    + gg.geom_abline(slope=1, intercept=0, color="red")
    + gg.labs(
        x="expr. in case-like capsules",
        y="expr. in non-case-like capsules",
        title="Target expression for guides inducing case-like phenotypes",
    )
)

In [ ]:
(
    gg.ggplot(plot_df, gg.aes(x="x_cp10k_phenotype", y="x_cp10k_nophenotype"))
    + gg.geom_point()
    + gg.theme_minimal()
    + gg.geom_abline(slope=1, intercept=0, color="red")
    + gg.labs(
        x="CP10k, case-like capsules",
        y="CP10k, non-case-like capsules",
        title="Target expression (CP10k) for guides inducing case-like phenotypes",
    )
)

## biological and technical factors

### External data loading

We first double-check that the guides we detect with phenotypes are properly filtered

In [ ]:
adata_all = sc.read_h5ad(
    "/workspace/data/251117_genomescale_CRISPRi/251117_genomescale_CRISPRi_biorep1_merged.h5ad"
)
new_obs = pd.read_pickle(
    "/workspace/data/251117_genomescale_CRISPRi/251117_genomescale_CRISPRi_biorep1_merged.genotyped.pkl"
)
adata_all.obs = new_obs
# adata_all_ = adata_all.copy()
# sc.pp.filter_cells(adata_all_, min_counts=10)

In [ ]:
adata_all.obs["library_size"] = adata_all.layers["reads"].sum(axis=1).A1
library_info = adata_all.obs.groupby("spacer")["library_size"].agg(
    library_size_mean="mean", library_size_median="median"
)

In [ ]:
def characterize_invalid_capsules(obs):
    # print(obs["library_size"] <= 200)
    # print(obs["library_size"])
    n_failing_capsules = 1.0 * (obs["library_size"] <= 200.0).sum()
    min_library_size = obs["library_size"].min()
    prop_failing_capsules = n_failing_capsules / len(obs)
    return pd.Series(
        {
            "n_failing_capsules": n_failing_capsules,
            "prop_failing_capsules": prop_failing_capsules,
            "min_library_size": min_library_size,
        }
    )


missing_capsules = (
    adata_all.obs.groupby("spacer")
    .apply(characterize_invalid_capsules)
    .sort_values("prop_failing_capsules")
)

In [ ]:
# library_size = adata_all_.layers["reads"].sum(axis=1)
# adata_all_.obs["library_size"] = library_size
# adata_all_.obs.groupby("spacer")["library_size"].agg(
#     {
#         "mean": "mean",
#         "median": "median",
#     }
# )

In [ ]:
library_info

In [ ]:
library_info

In [ ]:
spacer_trans_phenotypes_with_fitness = (
    spacer_trans_phenotypes.merge(fitness_df, left_on="spacer", right_index=True, how="right")
    .fillna(
        {
            "predictability_score": 0,
            "n_capsules_case_like": 0,
            "n_capsules_control_like": 0,
            "n_capsules": 0,
        }
    )
    .assign(
        spacer_induces_phenotype=lambda x: x["n_capsules_case_like"] >= 2,
        lateness=lambda x: (x["T4"] - x["T3"]) - (x["T2"] - x["T1"]),
    )
    .merge(library_info, left_on="spacer", right_index=True, how="left")
)
spacer_trans_phenotypes_with_fitness.assign(
    source="/workspace/experiments/01022026_multimodal/transcriptomic_analysis.ipynb"
).to_csv("spacer_predictability.csv", index=False)
spacer_trans_phenotypes_with_fitness

In [ ]:
spacer_trans_phenotypes_with_fitness["library_size_mean"].min()

### Spacer-level analysis

In [ ]:
q_threshold = 0.8
val_threshold = spacer_trans_phenotypes_with_fitness.query("spacer_induces_phenotype")[
    "T4"
].quantile(q_threshold)

(
    gg.ggplot(
        spacer_trans_phenotypes_with_fitness, gg.aes(x="T4", color="spacer_induces_phenotype")
    )
    + gg.stat_ecdf()
    + gg.geom_vline(xintercept=val_threshold, color="black")
    + gg.theme_minimal()
    + gg.labs(
        x="fitness (Calvo-Villamañán et al., 2020)", y="ECDF", color="induces case-like phenotype"
    )
    + gg.theme(
        legend_position="bottom",
        figure_size=(4, 3),
        axis_title=gg.element_text(size=7),
        axis_text=gg.element_text(size=6),
        legend_text=gg.element_text(size=6),  # Sets legend labels to 6pt
        legend_title=gg.element_text(size=6),  # Sets legend title to 6pt
    )
)

In [ ]:
spacer_trans_phenotypes_with_fitness["is_true_essential"] = (
    spacer_trans_phenotypes_with_fitness["T4"] <= val_threshold
)
spacer_trans_phenotypes_with_fitness["guide_type"] = np.where(
    (spacer_trans_phenotypes_with_fitness["is_true_essential"])
    & (spacer_trans_phenotypes_with_fitness["n_capsules_case_like"] >= 2),
    "detected_essential",
    np.where(
        spacer_trans_phenotypes_with_fitness["is_true_essential"], "missed_essential", "other"
    ),
)

In [ ]:
(
    gg.ggplot(
        spacer_trans_phenotypes_with_fitness,
        gg.aes(x="guide_type", y="T1"),
    )
    + gg.geom_violin()
    + gg.geom_jitter(alpha=0.1)
    + gg.theme_minimal()
)

In [ ]:
(
    gg.ggplot(
        spacer_trans_phenotypes_with_fitness,
        gg.aes(x="guide_type", y="n_capsules"),
    )
    # + gg.scale_y_log10()
    + gg.geom_violin()
    + gg.scale_y_continuous(limits=(0, 10))
    + gg.geom_jitter(alpha=0.1)
    + gg.theme_minimal()
)

In [ ]:
(
    gg.ggplot(
        spacer_trans_phenotypes_with_fitness.query("guide_type == 'missed_essential'"),
        gg.aes(x="n_capsules"),
    )
    + gg.stat_ecdf()
    + gg.theme_minimal()
)

In [ ]:
(
    gg.ggplot(
        spacer_trans_phenotypes_with_fitness,
        gg.aes(x="T1", color="guide_type"),
    )
    + gg.stat_ecdf()
    + gg.theme_minimal()
)

In [ ]:
for gene in (
    spacer_trans_phenotypes_with_fitness.query("guide_type == 'missed_essential'")
    .loc[lambda x: x["T1"] <= -2.5]["gene"]
    .value_counts()
    .index
):
    print(gene)

In [ ]:
(
    gg.ggplot(
        spacer_trans_phenotypes_with_fitness.query("guide_type == 'missed_essential'"),
        gg.aes(x="factor(n_capsules)", y="T1"),
    )
    + gg.geom_boxplot(outlier_shape=None)
    + gg.geom_jitter(alpha=0.1)
    + gg.theme_minimal()
)

In [ ]:
(
    gg.ggplot(
        spacer_trans_phenotypes_with_fitness,
        gg.aes(x="guide_type", y="library_size_mean"),
    )
    # + gg.scale_y_log10()
    + gg.geom_violin()
    + gg.geom_jitter(alpha=0.1)
    + gg.theme_minimal()
)

In [ ]:
(
    gg.ggplot(
        spacer_trans_phenotypes_with_fitness.query("guide_type == 'missed_essential'"),
        gg.aes(x="factor(n_capsules)", y="library_size_mean"),
    )
    + gg.geom_boxplot()
    # + gg.geom_jitter(alpha=0.1)
    # + gg.scale_y_log10()
    + gg.geom_hline(yintercept=200, color="red")
    + gg.theme_minimal()
)

In [ ]:
missed_essential_genes = (
    spacer_trans_phenotypes_with_fitness.query("guide_type == 'missed_essential'")["gene"]
    .value_counts()
    .loc[lambda x: x >= 4]
    #     .index
)
# for gene in missed_essential_genes.sort_values():
#     print(gene)

In [ ]:
spacer_trans_phenotypes_with_fitness.query("guide_type == 'missed_essential'")

### Gene-level analysis

In [ ]:
essential_genes = (
    spacer_trans_phenotypes_with_fitness.groupby("gene")["T4"].mean().loc[lambda x: x <= -3.0]
)

In [ ]:
genes_props = []
adata_ctrl = adata[adata.obs["gene"].astype(str).str.startswith("Control")]
adata.obs["target_"] = adata.obs["target"].astype(str)
for gene in tqdm(adata.obs["target_"].unique()):
    if gene == "nan":
        continue
    n_capsules_with_phenotype = (
        adata.obs.query("target == @gene")["annotated_cluster"].value_counts().get("case-like", 0)
    )
    adata_gene = adata[adata.obs["target"] == gene]

    # compute n DEGs
    pvals = st.ttest_ind(
        adata_gene.layers["cp10k"].toarray(),
        adata_ctrl.layers["cp10k"].toarray(),
        equal_var=False,
        axis=0,
    ).pvalue
    pvals[np.isnan(pvals)] = 1.0
    padjs = multipletests(pvals, method="fdr_bh")[1]
    n_degs = (padjs < 0.05).sum()

    genes_props.append(
        {
            "gene": gene,
            "n_capsules_with_phenotype": n_capsules_with_phenotype,
            "n_capsules": adata.obs.query("target == @gene").shape[0],
            "is_essential": gene in essential_genes.index,
            "is_detected_transcriptomic": n_capsules_with_phenotype >= 2.0,
            "n_degs": n_degs,
        }
    )
genes_props = pd.DataFrame(genes_props)

genes_props.assign(
    source="/workspace/experiments/01022026_multimodal/transcriptomic_analysis.ipynb"
).to_csv("gene_predictability.csv", index=False)

In [ ]:
genes_props["is_detected_transcriptomic"].sum(), genes_props["is_essential"].sum()

In [ ]:
print(", ".join(genes_props.query("is_essential")["gene"]))

In [ ]:
gene_properties = {
    "ispG": "Metabolism (Cofactors)",
    "zipA": "Cell Division",
    "rimP": "Ribosome Biogenesis",
    "rluD": "Ribosome Biogenesis",
    "obgE": "Ribosome Biogenesis",
    "yceD": "Ribosome Biogenesis",
    "ftsK": "Cell Division",
    "orn": "RNA Degradation",
    "der": "Ribosome Biogenesis",
    "cydD": "Aerobic Respiration",
    "higA": "Toxin-Antitoxin System",
    "lolA": "LPS Biosynthesis",
    "hemD": "Metabolism (Cofactors)",
    "rfaE": "LPS Biosynthesis",
    "iscR": "Metabolism (Cofactors)",
    "ptsI": "PTS System",
    "mukB": "DNA Replication",
    "gyrA": "DNA Replication",
    "ispF": "Metabolism (Cofactors)",
    "dapB": "Cell Wall",
    "pdxJ": "Metabolism (Cofactors)",
    "rimM": "Ribosome Biogenesis",
    "trmD": "Translation Elongation & tRNA Charging",
    "rpmI": "Ribosome Biogenesis",
    "dapA": "Cell Wall",
    "rpmD": "Ribosome Biogenesis",
    "lexA": "DNA Replication",
    "ftsQ": "Cell Division",
    "accC": "Metabolism (Fatty Acids)",
    "lapB": "LPS Biosynthesis",
    "ftsY": "Protein Secretion",
    "cca": "Translation Elongation & tRNA Charging",
    "yaaY": "Unknown",
    "polA": "DNA Replication",
    "rbfA": "Ribosome Biogenesis",
    "rpsU": "Ribosome Biogenesis",
    "ptsH": "PTS System",
    "crr": "PTS System",
    "hemE": "Metabolism (Cofactors)",
    "yejL": "LPS Biosynthesis",
    "nrdR": "DNA Replication",
    "hemB": "Metabolism (Cofactors)",
    "metY": "Translation Elongation & tRNA Charging",
    "rpsE": "Ribosome Biogenesis",
    "dfp": "Metabolism (Cofactors)",
    "lgt": "LPS Biosynthesis",
    "acpS": "Metabolism (Fatty Acids)",
    "rplN": "Ribosome Biogenesis",
    "mraY": "Cell Wall",
    "rpmB": "Ribosome Biogenesis",
    "mrdA": "Cell Wall",
    "parC": "DNA Replication",
    "priA": "DNA Replication",
    "lipA": "Metabolism (Cofactors)",
    "surA": "Cell Envelope Biogenesis",
    "plsC": "Metabolism (Fatty Acids)",
    "guaB": "Metabolism (Nucleotides)",
    "rne": "RNA Degradation",
    "wzyE": "LPS Biosynthesis",
    "secM": "Protein Secretion",
    "nusA": "Transcription Termination",
    "pcnB": "RNA Processing",
    "nhaA": "Cell Envelope Biogenesis",
    "hflD": "Protein Degradation",
    "ubiH": "Metabolism (Cofactors)",
    "gapA": "Metabolism (Central)",
    "lptF": "LPS Biosynthesis",
    "rpsI": "Ribosome Biogenesis",
    "frr": "Ribosome Biogenesis",
    "parE": "DNA Replication",
    "tufB": "Translation Elongation & tRNA Charging",
    "ppiB": "Protein Folding",
    "rpoE": "Transcription",
    "pssA": "Cell Envelope Biogenesis",
    "serS": "Translation Elongation & tRNA Charging",
    "leuU": "Translation Elongation & tRNA Charging",
    "mreD": "Cell Wall",
    "rplI": "Ribosome Biogenesis",
    "phoU": "Phosphate Transport",
    "tufA": "Translation Elongation & tRNA Charging",
    "rpsM": "Ribosome Biogenesis",
    "pstA": "Phosphate Transport",
    "hfq": "RNA Processing",
    "metG": "Translation Elongation & tRNA Charging",
    "tff": "Unknown",
    "rrsD": "Ribosome Biogenesis",
    "topA": "DNA Replication",
    "ubiC": "Metabolism (Cofactors)",
    "coaA": "Metabolism (Cofactors)",
    "ubiE": "Metabolism (Cofactors)",
    "rpoD": "Transcription",
    "iscU": "Metabolism (Cofactors)",
    "ubiB": "Metabolism (Cofactors)",
    "tmk": "Metabolism (Nucleotides)",
    "rpsP": "Ribosome Biogenesis",
    "thrV": "Translation Elongation & tRNA Charging",
    "rpsB": "Ribosome Biogenesis",
    "lolE": "LPS Biosynthesis",
    "arcA": "Aerobic Respiration",
    "nusG": "Transcription Termination",
    "dnaE": "DNA Replication",
    "ispH": "Metabolism (Cofactors)",
    "rpsO": "Ribosome Biogenesis",
    "rnc": "Ribosome Biogenesis",
    "tusC": "Translation Elongation & tRNA Charging",
    "cra": "Carbon Storage Regulation",
    "rnpB": "RNA Processing",
    "guaA": "Metabolism (Nucleotides)",
    "crp": "Transcription",
    "rplO": "Ribosome Biogenesis",
    "nusB": "Ribosome Biogenesis",
    "valS": "Translation Elongation & tRNA Charging",
    "cydB": "Aerobic Respiration",
    "gltX": "Translation Elongation & tRNA Charging",
    "pyrH": "Metabolism (Nucleotides)",
    "rpmA": "Ribosome Biogenesis",
    "lptA": "LPS Biosynthesis",
    "mreC": "Cell Wall",
    "coaD": "Metabolism (Cofactors)",
    "skp": "Cell Envelope Biogenesis",
    "groL": "Protein Folding",
    "rsgA": "Ribosome Biogenesis",
    "glmM": "Cell Wall",
    "fabH": "Metabolism (Fatty Acids)",
    "pgk": "Metabolism (Central)",
    "rlmE": "Ribosome Biogenesis",
    "rpmE": "Ribosome Biogenesis",
    "dcd": "Metabolism (Nucleotides)",
    "leuW": "Translation Elongation & tRNA Charging",
    "trpT": "Translation Elongation & tRNA Charging",
    "secB": "Protein Secretion",
    "yqgF": "Unknown",
    "pgsA": "Cell Envelope Biogenesis",
    "lpcA": "LPS Biosynthesis",
    "yrfF": "Cell Envelope Biogenesis",
    "iscS": "Metabolism (Cofactors)",
    "acnB": "Metabolism (Central)",
    "suhB": "Ribosome Biogenesis",
    "lpxH": "LPS Biosynthesis",
    "yejM": "LPS Biosynthesis",
    "ribD": "Metabolism (Cofactors)",
    "ddlB": "Cell Wall",
    "dapD": "Cell Wall",
    "waaG": "LPS Biosynthesis",
    "lptD": "LPS Biosynthesis",
    "metK": "Metabolism (Cofactors)",
    "prmC": "Ribosome Biogenesis",
    "kdsD": "LPS Biosynthesis",
    "aceE": "Aerobic Respiration",
    "ftsZ": "Cell Division",
    "map": "Protein Modification",
    "minE": "Cell Division",
    "bamA": "Cell Envelope Biogenesis",
    "valW": "Translation Elongation & tRNA Charging",
    "secG": "Protein Secretion",
    "rsmH": "Ribosome Biogenesis",
    "fabD": "Metabolism (Fatty Acids)",
    "glnS": "Translation Elongation & tRNA Charging",
    "waaC": "LPS Biosynthesis",
    "dxr": "Metabolism (Cofactors)",
    "ftsN": "Cell Division",
    "rpsN": "Ribosome Biogenesis",
    "kdsC": "LPS Biosynthesis",
    "waaF": "LPS Biosynthesis",
    "era": "Ribosome Biogenesis",
    "ibaG": "Cell Envelope Biogenesis",
    "secE": "Protein Secretion",
    "fabA": "Metabolism (Fatty Acids)",
    "fabI": "Metabolism (Fatty Acids)",
    "ffs": "Protein Secretion",
    "fmt": "Translation Elongation & tRNA Charging",
    "rplU": "Ribosome Biogenesis",
    "rplY": "Ribosome Biogenesis",
    "fusA": "Translation Elongation & tRNA Charging",
    "ffh": "Protein Secretion",
    "pheS": "Translation Elongation & tRNA Charging",
    "thrS": "Translation Elongation & tRNA Charging",
    "yihA": "Ribosome Biogenesis",
    "tusE": "Translation Elongation & tRNA Charging",
    "rho": "Transcription Termination",
    "lpxL": "LPS Biosynthesis",
    "argX": "Translation Elongation & tRNA Charging",
    "pth": "Translation Elongation & tRNA Charging",
    "pheT": "Translation Elongation & tRNA Charging",
    "tsaC": "Translation Elongation & tRNA Charging",
    "lptG": "LPS Biosynthesis",
    "lnt": "LPS Biosynthesis",
    "rpsC": "Ribosome Biogenesis",
    "hns": "Transcription",
    "rlmH": "Ribosome Biogenesis",
    "dut": "Metabolism (Nucleotides)",
    "rplP": "Ribosome Biogenesis",
    "secD": "Protein Secretion",
    "rpsJ": "Ribosome Biogenesis",
    "rplM": "Ribosome Biogenesis",
    "rplF": "Ribosome Biogenesis",
    "tsaB": "Translation Elongation & tRNA Charging",
    "rplX": "Ribosome Biogenesis",
    "ycaR": "Unknown",
    "pepP": "Protein Degradation",
    "ubiG": "Metabolism (Cofactors)",
    "waaQ": "LPS Biosynthesis",
    "pstB": "Phosphate Transport",
    "ypaB": "Unknown",
    "fabB": "Metabolism (Fatty Acids)",
    "rpsH": "Ribosome Biogenesis",
    "hemC": "Metabolism (Cofactors)",
    "ftsA": "Cell Division",
    "nadK": "Metabolism (Cofactors)",
    "tusD": "Translation Elongation & tRNA Charging",
    "rplE": "Ribosome Biogenesis",
    "rplK": "Ribosome Biogenesis",
    "ftsB": "Cell Division",
    "fldA": "Metabolism (Cofactors)",
    "serV": "Translation Elongation & tRNA Charging",
    "rplD": "Ribosome Biogenesis",
    "adk": "Metabolism (Nucleotides)",
    "ileS": "Translation Elongation & tRNA Charging",
    "lpxK": "LPS Biosynthesis",
    "rpsL": "Ribosome Biogenesis",
    "holA": "DNA Replication",
    "tsf": "Translation Elongation & tRNA Charging",
    "cysS": "Translation Elongation & tRNA Charging",
    "racR": "DNA Replication",
    "rrlC": "Ribosome Biogenesis",
    "lepB": "Protein Secretion",
    "rplB": "Ribosome Biogenesis",
    "gpsA": "Cell Envelope Biogenesis",
    "ymfK": "Cell Division",
    "asnS": "Translation Elongation & tRNA Charging",
    "rplC": "Ribosome Biogenesis",
    "mtn": "Metabolism (Cofactors)",
    "prs": "Metabolism (Nucleotides)",
    "yidD": "Cell Envelope Biogenesis",
    "nrdA": "DNA Replication",
    "rpsA": "Ribosome Biogenesis",
    "yheO": "Cell Envelope Biogenesis",
    "mreB": "Cell Wall",
    "ftsI": "Cell Division",
    "accB": "Metabolism (Fatty Acids)",
    "hda": "DNA Replication",
    "ssrA": "RNA Processing",
    "rpmJ": "Ribosome Biogenesis",
    "thyA": "DNA Replication",
    "secF": "Protein Secretion",
    "hemH": "Metabolism (Cofactors)",
    "murB": "Cell Wall",
    "rnpA": "RNA Processing",
    "rplW": "Ribosome Biogenesis",
    "rsfS": "Ribosome Biogenesis",
    "ftsH": "Cell Envelope Biogenesis",
    "xseB": "DNA Replication",
    "lpxA": "LPS Biosynthesis",
    "dnaB": "DNA Replication",
    "valV": "Translation Elongation & tRNA Charging",
    "prfA": "Translation Elongation & tRNA Charging",
    "hisS": "Translation Elongation & tRNA Charging",
    "rrfA": "Ribosome Biogenesis",
    "glyT": "Translation Elongation & tRNA Charging",
    "ispB": "Metabolism (Cofactors)",
    "cydC": "Aerobic Respiration",
    "dnaG": "DNA Replication",
    "lolC": "LPS Biosynthesis",
    "pnp": "RNA Degradation",
    "priB": "DNA Replication",
    "holB": "DNA Replication",
    "rpmF": "Ribosome Biogenesis",
    "csrA": "Carbon Storage Regulation",
    "rhoL": "Transcription Termination",
    "accD": "Metabolism (Fatty Acids)",
    "thrT": "Translation Elongation & tRNA Charging",
    "kdsA": "LPS Biosynthesis",
    "folC": "Metabolism (Cofactors)",
    "dnaK": "Protein Folding",
    "folA": "Metabolism (Cofactors)",
    "dxs": "Metabolism (Cofactors)",
    "ybeY": "Ribosome Biogenesis",
    "lpxD": "LPS Biosynthesis",
    "rpsG": "Ribosome Biogenesis",
    "glmU": "Cell Wall",
    "cydA": "Aerobic Respiration",
    "mukF": "DNA Replication",
    "ribA": "Metabolism (Cofactors)",
    "fabG": "Metabolism (Fatty Acids)",
    "lolD": "LPS Biosynthesis",
    "lpd": "Aerobic Respiration",
    "aceF": "Aerobic Respiration",
    "yqgE": "Unknown",
    "kdsB": "LPS Biosynthesis",
    "nadD": "Metabolism (Cofactors)",
    "rpsS": "Ribosome Biogenesis",
    "argS": "Translation Elongation & tRNA Charging",
    "murC": "Cell Wall",
    "leuZ": "Translation Elongation & tRNA Charging",
    "asmA": "LPS Biosynthesis",
    "dnaN": "DNA Replication",
    "groS": "Protein Folding",
    "dicA": "DNA Replication",
    "ppa": "Metabolism (Nucleotides)",
    "tyrU": "Translation Elongation & tRNA Charging",
    "ubiD": "Metabolism (Cofactors)",
    "ribF": "Metabolism (Cofactors)",
    "yhbE": "Ribosome Biogenesis",
    "rpmC": "Ribosome Biogenesis",
    "infB": "Ribosome Biogenesis",
    "prfB": "Translation Elongation & tRNA Charging",
    "serT": "Translation Elongation & tRNA Charging",
    "pyrG": "Metabolism (Nucleotides)",
    "infC": "Translation Elongation & tRNA Charging",
    "mnmA": "Translation Elongation & tRNA Charging",
    "rodZ": "Cell Wall",
    "cysE": "Metabolism (Amino Acids)",
    "lpxB": "LPS Biosynthesis",
    "secY": "Protein Secretion",
    "tyrS": "Translation Elongation & tRNA Charging",
    "fbaA": "Metabolism (Central)",
    "rpmH": "Ribosome Biogenesis",
    "secA": "Protein Secretion",
    "nrdB": "DNA Replication",
    "dnaX": "DNA Replication",
    "proS": "Translation Elongation & tRNA Charging",
    "rpoB": "Transcription",
    "gmk": "Metabolism (Nucleotides)",
    "hemA": "Metabolism (Cofactors)",
    "lysS": "Translation Elongation & tRNA Charging",
    "rpsF": "Ribosome Biogenesis",
    "rnhB": "RNA Degradation",
    "hemL": "Metabolism (Cofactors)",
    "msbA": "LPS Biosynthesis",
    "trpS": "Translation Elongation & tRNA Charging",
    "acpP": "Metabolism (Fatty Acids)",
    "glyQ": "Translation Elongation & tRNA Charging",
    "ribE": "Metabolism (Cofactors)",
    "ispD": "Metabolism (Cofactors)",
    "rpsT": "Ribosome Biogenesis",
    "rpoZ": "Transcription",
    "rplT": "Ribosome Biogenesis",
    "waaA": "LPS Biosynthesis",
    "folE": "Metabolism (Cofactors)",
    "murF": "Cell Wall",
    "tsaD": "Translation Elongation & tRNA Charging",
    "rpsR": "Ribosome Biogenesis",
    "tpke11": "Unknown",
    "rseP": "Cell Envelope Biogenesis",
    "lspA": "LPS Biosynthesis",
    "rnt": "RNA Processing",
    "rrsC": "Ribosome Biogenesis",
    "lepA": "Translation Elongation & tRNA Charging",
    "psd": "Cell Envelope Biogenesis",
    "fabZ": "Metabolism (Fatty Acids)",
    "pdhR": "Aerobic Respiration",
    "spoT": "Metabolism (Nucleotides)",
    "waaP": "LPS Biosynthesis",
    "sroE": "Unknown",
    "ribC": "Metabolism (Cofactors)",
    "mukE": "DNA Replication",
    "murA": "Cell Wall",
    "rplA": "Ribosome Biogenesis",
    "def": "Translation Elongation & tRNA Charging",
    "murI": "Cell Wall",
    "rrlH": "Ribosome Biogenesis",
    "purB": "Metabolism (Nucleotides)",
    "leuS": "Translation Elongation & tRNA Charging",
    "lpxC": "LPS Biosynthesis",
    "rplS": "Ribosome Biogenesis",
    "lptE": "LPS Biosynthesis",
    "lptB": "LPS Biosynthesis",
    "eno": "Metabolism (Central)",
    "ftsL": "Cell Division",
    "yidC": "Protein Secretion",
    "cydX": "Aerobic Respiration",
    "fkpB": "Protein Folding",
    "lolB": "LPS Biosynthesis",
    "nadE": "Metabolism (Cofactors)",
    "ubiX": "Metabolism (Cofactors)",
    "rplQ": "Ribosome Biogenesis",
    "gyrB": "DNA Replication",
    "infA": "Translation Elongation & tRNA Charging",
    "murD": "Cell Wall",
    "rfaD": "LPS Biosynthesis",
    "can": "Metabolism (Other)",
    "ispA": "Metabolism (Cofactors)",
    "alaS": "Translation Elongation & tRNA Charging",
    "dnaC": "DNA Replication",
    "rrsA": "Ribosome Biogenesis",
    "ftsW": "Cell Division",
    "ubiJ": "Metabolism (Cofactors)",
    "ispE": "Metabolism (Cofactors)",
    "proM": "Translation Elongation & tRNA Charging",
    "rplR": "Ribosome Biogenesis",
    "aspS": "Translation Elongation & tRNA Charging",
    "mraZ": "Cell Division",
    "tsaE": "Translation Elongation & tRNA Charging",
    "ispU": "Metabolism (Cofactors)",
    "rpoC": "Transcription",
    "cmk": "Metabolism (Nucleotides)",
    "plsB": "Metabolism (Fatty Acids)",
    "dnaT": "DNA Replication",
    "cdsA": "Cell Envelope Biogenesis",
    "rplV": "Ribosome Biogenesis",
    "rpmG": "Ribosome Biogenesis",
    "rpsQ": "Ribosome Biogenesis",
    "hisR": "Translation Elongation & tRNA Charging",
    "rrsG": "Ribosome Biogenesis",
    "ssb": "DNA Replication",
}

essential_genes_props = genes_props.query("is_essential").copy()
essential_genes_props["gene_annotation"] = essential_genes_props["gene"].map(gene_properties)
essential_genes_props.groupby("gene_annotation").size().sort_values()

coarse_gene_mapping = {
    "Ribosome Biogenesis": "Ribosome & Translation",
    "Translation Elongation & tRNA Charging": "Ribosome & Translation",
    "DNA Replication": "DNA Replication & Nucleotides",
    "Metabolism (Nucleotides)": "DNA Replication & Nucleotides",
    "Transcription": "Transcription & RNA",
    "Transcription Termination": "Transcription & RNA",
    "RNA Processing": "Transcription & RNA",
    "RNA Degradation": "Transcription & RNA",
    "Carbon Storage Regulation": "Transcription & RNA",
    "Cell Wall": "Cell Wall & Division",
    "Cell Division": "Cell Wall & Division",
    "LPS Biosynthesis": "LPS & Lipid Biosynthesis",
    "Cell Envelope Biogenesis": "LPS & Lipid Biosynthesis",
    "Metabolism (Fatty Acids)": "LPS & Lipid Biosynthesis",
    "Protein Secretion": "Protein Secretion & QC",
    "Protein Quality Control": "Protein Secretion & QC",
    "Protein Degradation": "Protein Secretion & QC",
    "Protein Folding": "Protein Secretion & QC",
    "Toxin-Antitoxin System": "Protein Secretion & QC",
    "Protein Modification": "Protein Secretion & QC",
    "Metabolism (Cofactors)": "Central Metabolism",
    "Aerobic Respiration": "Central Metabolism",
    "Metabolism (Central)": "Central Metabolism",
    "Metabolism (Amino Acids)": "Central Metabolism",
    "PTS System": "Central Metabolism",
    "Phosphate Transport": "Central Metabolism",
    "Metabolism (Other)": "Central Metabolism",
    "Unknown": "Other",
}

essential_genes_props["gene_annotation_coarse"] = essential_genes_props["gene_annotation"].map(
    coarse_gene_mapping
)

In [ ]:
errs_per_group = (
    essential_genes_props.groupby("gene_annotation_coarse")["is_detected_transcriptomic"]
    .mean()
    .sort_values()
    .to_frame("proportion detected")
    .reset_index()
    .loc[lambda x: x["gene_annotation_coarse"] != "Other"]
)

errs_per_group["gene_annotation_coarse"] = pd.Categorical(
    errs_per_group["gene_annotation_coarse"],
    categories=errs_per_group["gene_annotation_coarse"],
    ordered=True,
)

(
    gg.ggplot(
        errs_per_group,
        gg.aes(x="gene_annotation_coarse", y="proportion detected"),
    )
    + gg.geom_col(width=0.4)
    + gg.coord_flip()
    + gg.theme_minimal()
    + gg.theme(
        axis_title=gg.element_text(size=7),
        text=gg.element_text(size=6),
        figure_size=(4, 1),
    )
    + gg.labs(x="", y="% of essential genes detected")
)

In [ ]:
errs_per_group["gene_annotation_coarse"] = pd.Categorical(
    errs_per_group["gene_annotation_coarse"],
    categories=errs_per_group["gene_annotation_coarse"],
    ordered=True,
)

In [ ]:
errs_per_group

In [ ]:
from matplotlib_venn import venn2

venn2(
    [
        set(genes_props.query("is_detected_transcriptomic")["gene"]),
        set(genes_props.query("is_essential")["gene"]),
    ],
    set_labels=("detected via transcriptomics", "essential"),
)

In [ ]:
non_detected_essential_genes = (
    genes_props.query("~is_detected_transcriptomic").query("is_essential")["gene"].sort_values()
)
print(", ".join(non_detected_essential_genes))

In [ ]:
(
    gg.ggplot(
        genes_props.query("is_essential"),
        gg.aes(x="is_detected_transcriptomic", y="n_degs"),
    )
    + gg.geom_violin()
    + gg.scale_y_log10()
    + gg.theme_minimal()
    + SHARED_THEME
    + gg.theme(figure_size)
    + gg.labs(x="is the essential gene detected?", y="# of DEGs")
)

non-detected essential genes still have a large number of DEGs. Would an approach that directly leverages sample-level information be more effective?

- see `/workspace/experiments/01022026_multimodal/transcriptomic_recluster.ipynb`
- see `/workspace/experiments/01022026_multimodal/transcriptomic_mrvi.ipynb`

In [ ]:
f_vector_cols = [
    # "N Mismatch_y",
    "Delta time (s)",
    "Instantaneous Growth Rate: Volume",
    "Length",
    "Septum Displacement Length Normalized",
    "Width",
    "mCherry mean_intensity",
]


timeseries_df = pd.read_pickle(
    "/workspace/data/Eaton_2025/Data/lDE20_Imaging/Clustering/2023-01-23_sgRNA_Timeseries_df.pkl"
)
working_dir = "/workspace/data/Eaton_2025/Data/lDE20_Imaging"
df = pd.read_csv(
    os.path.join(working_dir, "2024-01-25_lDE20_Steady_State_df_Estimators_wStats.csv")
)

df_ = df.query("Estimator == 'Mean (Robust)'")
ops_summarystats_df = (
    df_.pivot(index=["sgRNA", "Gene", "N Mismatch"], columns="Variable(s)", values="Value")
    .reset_index()
    .set_index(["sgRNA", "Gene"])
)
ops_summarystats_df


def expand_embeddings(df, columns):
    expanded_dfs = []
    for col in columns:
        if col not in df.columns:
            continue
        col_values = np.stack(df[col].values)
        if col_values.ndim > 2:
            col_values = col_values.reshape(len(df), -1)

        expanded = pd.DataFrame(
            col_values, index=df.index, columns=[f"{col}_{i}" for i in range(col_values.shape[1])]
        )
        expanded_dfs.append(expanded)

    metadata_cols = [c for c in df.columns if c not in columns]
    return pd.concat([df[metadata_cols]] + expanded_dfs, axis=1).reset_index()


df_embeddings = expand_embeddings(timeseries_df, ["Feature Vector"])
tsne_rep = TSNE(n_components=2)
embeddings_tsne = tsne_rep.fit_transform(
    df_embeddings[[c for c in df_embeddings.columns if c.startswith("Feature Vector")]]
)
df_embeddings["tsne_x"] = embeddings_tsne[:, 0]
df_embeddings["tsne_y"] = embeddings_tsne[:, 1]
df_embeddings.info()

pca_ = PCA(n_components=2)
pca_.fit(df_embeddings[[c for c in df_embeddings.columns if c.startswith("Feature Vector")]])
embeddings_pca = pca_.transform(
    df_embeddings[[c for c in df_embeddings.columns if c.startswith("Feature Vector")]]
)
df_embeddings["pca_x"] = embeddings_pca[:, 0]
df_embeddings["pca_y"] = embeddings_pca[:, 1]
# comparison with growth fitness screen

ops_df = df_embeddings.merge(ops_summarystats_df.reset_index(), on="sgRNA", how="left")
# ops_df

In [ ]:
# spacer_trans_phenotypes_with_fitness
# guide-type
# identify and visualize control guide
# identify and visualize detected and non-detected guides

In [ ]:
pd.set_option("display.max_columns", 500)

In [ ]:
spacer_trans_phenotypes_with_fitness

In [ ]:
joint_df = ops_df.merge(
    spacer_trans_phenotypes_with_fitness, left_on="sgRNA", right_on="spacer", how="left"
)
joint_df_overlap = joint_df.loc[lambda x: ~x["guide_type"].isna()]
print(joint_df_overlap.shape, spacer_trans_phenotypes_with_fitness.shape)

In [ ]:
(
    gg.ggplot(
        joint_df,
        gg.aes(x="tsne_x", y="tsne_y"),
    )
    + gg.geom_point(size=0.1)
    + gg.geom_point(
        joint_df_overlap,
        gg.aes(x="tsne_x", y="tsne_y", color="guide_type"),
        size=1.0,
    )
    + gg.theme_minimal()
)